<a href="https://colab.research.google.com/github/RatchapolSamalee/Dunnhumby-SQL-PYTHON-transactions-analysis/blob/main/600659_Mini_Hackathon_1_Data_(SuperAI_SS_6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Hackathon 1: Data โดยผู้เข้าร่วมหมายเลข 600659

---


## **ข้อมูลผู้ submit**

นายรัชพล สม่าหลี นักศึกษาสาขาสถิติ ชั้นปีที่ 4 คณะวิทยาศาสตร์และเทคโนโลยี สาขาสถิติ มหาวิทยาลัยธรรมศาสตร์

**สาเหตุที่เลือกชุดข้อมูลนี้** :

จากการตรวจสอบเบื้องต้น พบว่าข้อมูลนี้มีการเตรียมข้อมูลที่ค่อนข้างท้ายทาย ทั้งการจัดการกับ missing values เมื่อปรับเป็น dataframe,  การจัดการกับข้อมูลไม่สอคคล้อง (เช่น มีการผสมเว้นวรรคเข้ามาใน value ของข้อมูล) นอกจากนี้เนื่องจากข้อมูลเป็นข้อมูลอนุกรมเวลา ทำให้การวิเคราะห์จะมีการใช้สถิติที่แตกต่างออกไปจากข้อมูลตัดขวาง ทำให้ข้อมูลนี้มีความน่าสนใจในการศึกษาหามุมมองในการวิเคราะห์ใหม่ ๆ


## **คำแนะนำ:**

ผมได้จัด Markdownตาม Headline แล้วสามารถใช้ Table of Contents ของ Colab ในการเป็น Guide (ดูและกด) ระหว่างการตรวจได้ครับเพื่อความสะดวก

<br>

---

# การวิเคราะห์ข้อมูลการใช้ไฟฟ้า - 18 เขตให้บริการของการไฟฟ้านครหลวง

---

## ภาพรวมชุดข้อมูล

| รายการ | รายละเอียด |
|--------|------------|
| แหล่งข้อมูล | การไฟฟ้านครหลวง (MEA) |
| ช่วงเวลา | ปี 2562-2568 (7 ปี รายเดือน) |
| จำนวนแถว | 83 แถว (เดือน) |
| จำนวนเขต | 18 เขตจำหน่ายในกรุงเทพและปริมณฑล |

## โครงสร้าง Notebook
Prepare 1: Setup และโหลดข้อมูล<br>
Prepare 2: ทำความสะอาดและเตรียมข้อมูลก่อนการวิเคราะห์<br>
EDA: สำรวจลักษณะของข้อมูหลังทำความสะอาด<br>
1. คำถามที่ 1 - การใช้ไฟฟ้ารวมเปลี่ยนแปลงอย่างไรใน 7 ปี Trend & Anomaly Detection
2. คำถามที่ 2 - เขตไหนใช้ไฟมากที่สุด และใครโตเร็วที่สุด District Ranking & Growth
3. คำถามที่ 3 - รูปแบบการใช้ไฟฟ้าตามฤดูกาลต่างกันระหว่างเขตไหม Seasonal Pattern by District
4. การวิเคราะห์ข้อมูลเพิ่มเติม
5. สรุป finding และ Actions


## <u>Prepare 1:</u> Setup และโหลดข้อมูล


In [ ]:
# นำเข้า Library ทั้งหมดและตั้งค่าฟอนต์ภาษาไทย

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import os
from scipy import stats

warnings.filterwarnings('ignore')

# โหลด TH Sarabun New
FONT_FILE = 'thsarabunnew-webfont.ttf'
if not os.path.exists(FONT_FILE):
    os.system('wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf')
mpl.font_manager.fontManager.addfont(FONT_FILE)
mpl.rc('font', family='TH Sarabun New', size=24)
sns.set_theme(style='whitegrid', font='TH Sarabun New')
mpl.rc('axes', titlesize=24, labelsize=24)
mpl.rc('xtick', labelsize=24)
mpl.rc('ytick', labelsize=24)
mpl.rc('legend', title_fontsize=20, fontsize=20)

print('Font: TH Sarabun New ติดตั้งสำเร็จ')

Font: TH Sarabun New ติดตั้งสำเร็จ


In [ ]:
# โหลดไฟล์ Data เข้าสู่โน้ตบุคจาก Google Sheet
URL = 'https://docs.google.com/spreadsheets/d/1Im61yctLi487RObV6799B_j4rbvdWeADXWgjquM6rAM/export?format=csv&gid=1199395040'

df_raw = pd.read_csv(URL, encoding='utf-8-sig')

print("ภาพรวมข้อมูล")

print("\n[Column Details & Missing Values]")
info_df = pd.DataFrame({
    "Type": df_raw.dtypes,
    "Non-Null": df_raw.count(),
    "Missing": df_raw.isnull().sum(),
    "Unique": df_raw.nunique(),
    "Duplicates": df_raw.duplicated().sum()
})

display(info_df)

ภาพรวมข้อมูล

[Column Details & Missing Values]


,Type,Non-Null,Missing,Unique,Duplicates
ปี,float64,7,76,7,0
เดือน,object,83,0,16,0
วัดเลียบ,float64,83,0,82,0
คลองเตย,float64,83,0,82,0
ยานนาวา,float64,83,0,82,0
บางกะปิ,float64,83,0,83,0
มีนบุรี,float64,83,0,83,0
สมุทรปราการ,float64,83,0,82,0
บางพลี,float64,83,0,83,0
สามเสน,float64,83,0,83,0


- จากการตรวจสอบข้อมูลเบื้องต้น พบว่าชุดข้อมูลนี้เป็นการเก็บข้อมูลรูปแบบอนุกรมเวลา (Time Series) รายเดือน
- ไม่พบปัญหาข้อมูลแถวซ้ำ (Duplicated rows)
- ***Note*** พิจารณาไม่ทำการตรวจสอบ Outliers ของข้อมูลเนื่องข้อมูลอนุกรมเวลาจะเกิดช่วงเวลาขาดหายทำให้ข้อมูลอาจมีคุณภาพลดลง รวมถึงจากโจทย์ที่ได้รับมอบหมายจะต้องมีการวิเคราะห์ในช่วงเวลาผิดปกติด้วย
<br>
ขั้นตอนถัดไป

- คอลัมน์ `(หักไฟทำการ)` มีความคลุมเครือ จะต้องดำเนินการตรวจสอบ
- พบปัญหาข้อมูลมีค่าขาดหาย (Missing Values) ในคอลัมน์ปี จะต้องทำการแก้ไข
- ตรวจสอบความสม่ำเสมอ (Consistent) ของทั้งชุดข้อมูล
    - ข้อมูลที่เก็บมาต้องเป็นรูปแบบเดียวกันทั้งชุด
- ข้อมูลที่เก็บมาจะต้องถูกต้อง (>=0 ทุกจุดข้อมูล)



## <u>Prepare 2</u>: ทำความสะอาดและเตรียมข้อมูลก่อนการวิเคราะห์

ในขั้นตอนนี้จะมุ่งเน้นการจัดการปัญหาคุณภาพข้อมูลที่ตรวจพบข้างต้น โดยมีกระบวนการดังนี้:

<br>

### Prepare 2.1 **ตรวจสอบคอลัมน์ที่สร้างความสับสน** : คอลัมน์ `(หักไฟทำการ)`
  - ตรวจสอบว่า `(หักไฟทำการ)` สื่อถึงอะไร สมมติฐานคือสื่อถึงยอดรวมสะสมทุกเขตในเดือนนั้นๆ
  - ดำเนินการแปลงเป็น float แล้วตรวจสอบว่าเมื่อนำค่าจากทุกเขตในเดือนเดียวกันมารวมกัน มีค่าเท่ากับคอลัมน์ `(หักไฟทำการ)` เดือนนั้นหรือไม่
  - หากสมมติฐานถูกต้องจะทำการเปลี่ยนชื่อคอลัมน์เป็น `รวม` เพื่อให้เข้าใจได้ง่ายขึ้น

In [ ]:
df = df_raw.copy()
# ตรวจสอบว่า '(หักไฟทำการ)' เป็นข้อมูลประเภทอะไร
print('=== 1. ตรวจสอบ: คอลัมน์ (หักไฟทำการ) ===')
total_raw = df['(หักไฟทำการ)']
print(f'dtype เริ่มต้น : {total_raw.dtype}')
print(f'NaN    : {total_raw.isnull().sum()}')
print('ตัวอย่างค่า (5 แถวแรก):')
for v in total_raw.head(5):
    print(f'  {repr(v)}')

=== 1. ตรวจสอบ: คอลัมน์ (หักไฟทำการ) ===
dtype เริ่มต้น : object
NaN    : 0
ตัวอย่างค่า (5 แถวแรก):
  '4,119.31'
  '4,188.50'
  '4,660.94'
  '4,709.44'
  '4,976.37'


จากการตรวจสอบพบว่าข้อมูลมีลักษณะเป็น object ไม่ใช่ float ทำการแปลงเป็น float ก่อนเพื่อดูว่าหมายถึงยอดรวมทุกเขตจริงๆไหม

In [ ]:
# จากลักษณะคาดเดาว่าเป็นยอดรวมจึงทำการแปลงเป็น float เพือทดสอบว่าถูกต้องมั้ย
print("===แปลงค่าในคอลัมน์ 'หักไฟทำการ' เป็น float ===")
df['รวม'] = total_raw.str.replace(',', '', regex=False).astype(float)
print(f'dtype หลังแปลง : {df["รวม"].dtype}')
# เช็ค Null ก่อนจะได้ตรวจสอบทุกเดือน ไม่ตกหล่น
print(f'NaN   : {df["รวม"].isnull().sum()}')

===แปลงค่าในคอลัมน์ 'หักไฟทำการ' เป็น float ===
dtype หลังแปลง : float64
NaN   : 0


In [ ]:
# ดึงรายชื่อเขต (ชื่อคอลัมน์ทุกเขต) เพื่อใช้สำหรับทุกการวิเคราะห์ที่ต้องมีการนำชื่อเขตทั้งหมดไปใช้
DIST_COLS = df.columns[2:20]

In [ ]:
# ทดสอบว่า sum ของทุกเขต รวมกันได้เท่ากับคอลัมน์ '(หักไฟทำการ)' หรือไม่ โดยใช้การ Visualization บล็อกถัดไป
sum_districts = df[DIST_COLS].sum(axis=1)
diff = sum_districts - df['รวม']

fig_diff = px.histogram(diff.dropna(), nbins=20,
                        title='<b>การแจกแจงของความแตกต่างระหว่าง (หักไฟทำการ) และผลรวมของทุกเขตในเดือนเดียวกัน</b>',
                        color_discrete_sequence=['#1E88E5'])

fig_diff.update_layout(
    xaxis_title='<b>ค่าส่วนต่าง (ล้านหน่วย)</b>',
    yaxis_title='<b>ความถี่ (เดือน)</b>',
    font=dict(family="TH Sarabun New", size=20),
    title_font_size=24,
    showlegend=False,
    margin=dict(t=100)
)

fig_diff.show()

จากผลการตรวจสอบพบว่าโดยส่วนใหญ่แล้ว (>97%) `(หักไฟทำการ)` มีค่าเท่ากับผลรวมทุกเขต แถวที่มีค่าไม่เท่าก็แตกต่างน้อยมากๆจนไม่มีนัยสำคัญ ดังนั้นจึงสรุปว่า `(หักไฟทำการ)` มีความหมายคือผลรวมทุกๆเขตในแต่ละเดือน จะใช้คอลัมน์ `รวม` แทนคอลัมน์ `(หักไฟทำการ)` เพื่อลดความสับสนในการวิเคราะห์



In [ ]:
# ลบคอลัมน์ '(หักไฟทำการ)'
df.drop(columns=['(หักไฟทำการ)'], inplace=True)
df.head(5)

,ปี,เดือน,วัดเลียบ,คลองเตย,ยานนาวา,บางกะปิ,มีนบุรี,สมุทรปราการ,บางพลี,สามเสน,...,บางใหญ่,ธนบุรี,ราษฎร์บูรณะ,บางขุนเทียน,บางเขน,บางนา,บางบัวทอง,ลาดกระบัง,นวลจันทร์,รวม
0,2562.0,ม.ค.,93.96,375.86,154.62,243.49,156.70,400.83,409.56,291.46,...,134.90,152.69,321.13,199.88,239.95,220.28,146.70,152.37,150.26,4119.31
1,NaN,ก.พ.,94.88,373.54,156.55,248.90,161.21,396.36,408.68,298.58,...,142.73,158.55,314.98,202.95,254.82,222.29,152.47,155.39,158.49,4188.50
2,NaN,มี.ค.,103.27,412.09,170.67,272.89,180.84,445.57,458.59,328.44,...,159.21,174.84,361.92,225.96,281.99,246.60,172.70,167.90,174.00,4660.94
3,NaN,เม.ย.,108.59,419.71,179.36,293.99,188.01,400.70,426.82,345.59,...,173.20,187.59,338.99,224.96,295.08,255.97,176.43,172.99,187.62,4709.44
4,NaN,พ.ค.,110.97,429.76,184.89,307.18,195.85,446.77,468.74,355.00,...,176.45,193.47,368.12,243.04,307.90,268.79,186.15,194.68,193.52,4976.37


ดำเนินการลบคอลัมน์ `(หักไฟทำการ)` เพื่อใช้คอลัมน์ `รวม` แทน


<br>
<br>

### Prepare 2.2 การจัดการค่าว่าง (Missing Value ): เติมค่าว่างคอลัมน์ปีด้วย foward fill

In [ ]:
# ตรวจสอบก่อน: หา dtype, จำนวน NaN, ค่า unique
print('=== ตรวจสอบ: คอลัมน์ ปี ===')
print(f'dtype            : {df["ปี"].dtype}')
print(f'จำนวนแถวทั้งหมด : {len(df)}')
print(f'NaN              : {df["ปี"].isnull().sum()} แถว')
print(f'non-NaN          : {df["ปี"].notnull().sum()} แถว (1 แถวต่อ 1 ปี)')
print(f'ค่า unique        : {sorted(df["ปี"].dropna().unique())}')
print("\nแสดงตัวอย่างข้อมูล")
display(df[['ปี','เดือน']].head(14))

=== ตรวจสอบ: คอลัมน์ ปี ===
dtype            : float64
จำนวนแถวทั้งหมด : 83
NaN              : 76 แถว
non-NaN          : 7 แถว (1 แถวต่อ 1 ปี)
ค่า unique        : [np.float64(2562.0), np.float64(2563.0), np.float64(2564.0), np.float64(2565.0), np.float64(2566.0), np.float64(2567.0), np.float64(2568.0)]

แสดงตัวอย่างข้อมูล


,ปี,เดือน
0,2562.0,ม.ค.
1,NaN,ก.พ.
2,NaN,มี.ค.
3,NaN,เม.ย.
4,NaN,พ.ค.
5,NaN,มิ.ย.
6,NaN,ก.ค.
7,NaN,ส.ค.
8,NaN,ก.ย.
9,NaN,ต.ค.


Time Index ระบุปี (คอลัมน์ `ปี`) ยังมีค่าขาดหายตาม format ของข้อมูลต้นทาง ต้องมีการนำปีเดียวกันไปเติมค่าเดือนต่อไปอีก 11 เดือน (Forward Fill)

In [ ]:
# แก้ไขแถวที่ไม่มีค่า 'ปี' โดยการ forward fill แล้วเปลี่ยนชนิดข้อมูลเป็น int
df['ปี'] = df['ปี'].ffill().astype(int)
# ตรวจสอบผลลัพธ์
print()
print('=== ผลหลังทำความสะอาด ===')
print(f'dtype  : {df["ปี"].dtype}')
print(f'NaN    : {df["ปี"].isnull().sum()}')
print(f'unique : {sorted(df["ปี"].unique())}')
display(df[['ปี','เดือน']].head(14))


=== ผลหลังทำความสะอาด ===
dtype  : int64
NaN    : 0
unique : [np.int64(2562), np.int64(2563), np.int64(2564), np.int64(2565), np.int64(2566), np.int64(2567), np.int64(2568)]


,ปี,เดือน
0,2562,ม.ค.
1,2562,ก.พ.
2,2562,มี.ค.
3,2562,เม.ย.
4,2562,พ.ค.
5,2562,มิ.ย.
6,2562,ก.ค.
7,2562,ส.ค.
8,2562,ก.ย.
9,2562,ต.ค.



<br>
<br>

### Prepare 2.3 แก้ปัญหา Incosistent ใน แถวระบุเดือน: กำจัดอักขระพิเศษ (Whitespace) ออกจากชื่อเดือน

In [ ]:
# ตรวจสอบค่าที่เป็นไปได้ทั้งหมดในคอลัมน์เดือน
print(df['เดือน'].unique())

['ม.ค.' 'ก.พ.' 'มี.ค.' 'เม.ย.' 'พ.ค.' 'มิ.ย.' 'ก.ค.' 'ส.ค.' 'ก.ย.' 'ต.ค.'
 'พ.ย.' 'ธ.ค.' '\xa0ม.ค.\xa0' '\xa0ก.พ.\xa0' '\xa0มี.ค.\xa0'
 '\xa0เม.ย.\xa0']


ชื่อเดือนบางเดือนมี \xa0 ติดมาด้วยทำให้ชื่อเดือนเดียวกันถูกมองว่าคนละเดือนกัน ต้องได้รับการแก้ไข

In [ ]:
# ลบค่า Whitespace ออกจากชื่อเดือน
df['เดือน'] = df['เดือน'].str.replace('\xa0', '')

# ตรวจสอบค่าที่เป็นไปได้ทั้งหมดในคอลัมน์เดือนหลังลบ white space
print(df['เดือน'].unique())

['ม.ค.' 'ก.พ.' 'มี.ค.' 'เม.ย.' 'พ.ค.' 'มิ.ย.' 'ก.ค.' 'ส.ค.' 'ก.ย.' 'ต.ค.'
 'พ.ย.' 'ธ.ค.']


ทุกเดือนได้รับการแก้ไขปัญหา Whitespace แล้ว ตอนนี้ unique เหลือแค่ 12 ซึ่งถูกต้องตามที่ควรจะเป็น

<br>
<br>

### Prepare 2.4 ตรวจสอบปัญหา Incosistent ในคอลัมน์ที่ระบุเขต: ตรวจสอบอักษระพิเศษเช่นอาจมีช่องว่างปนเข้ามาในชื่อเขต (อาจเกิดกรณีเดียวกันกับชื่อเดือน)


ตรวจสอบว่ากรณีชื่อเขตมีเคสเดียวกันเกิดขึ้นหรือไม่

In [ ]:
# ตรวจสอบว่ามีช่องว่าง (Whitespace) อยู่ในชื่อคอลัมน์หรือไม่
has_space = [' ' in c for c in DIST_COLS]

# สร้าง DataFrame สำหรับตรวจสอบผล
check_df = pd.DataFrame({
    'column_name': [repr(c) for c in DIST_COLS],
    'has_space': has_space
})

# แสดงผลการตรวจสอบ
display(check_df)

# สรุปจำนวนคอลัมน์ที่มีช่องว่าง
print(f"คอลัมน์ที่มีช่องว่าง: {sum(has_space)} คอลัมน์")

,column_name,has_space
0,'วัดเลียบ',False
1,'คลองเตย',False
2,'ยานนาวา',False
3,'บางกะปิ',False
4,'มีนบุรี',False
5,'สมุทรปราการ',False
6,'บางพลี',False
7,'สามเสน',False
8,'นนทบุรี',False
9,'บางใหญ่',False


คอลัมน์ที่มีช่องว่าง: 0 คอลัมน์


ในส่วนของชื่อเขต ไม่พบว่าเกิดปัญหา whitespace ติดมากับข้อความ

<br>
<br>

### Prepare 2.5 การตรวจสอบความถูกต้องของข้อมูลปริมาณการใช้ไฟฟ้าที่เก็บรวบรวมมา <br> (อาจมี Inconsistent data point ที่มีค่าติดลบ) : ตรวจสอบว่าทุกจุดข้อมูลที่เก็บมา จะต้องมีค่า **>= 0**

In [ ]:
# ฟังก์ชั่นตรวจสอบค่าติดลบ
def check_if_negative_exists(dataframe, column_name):
    return dataframe[column_name].min() < 0

In [ ]:
# สร้าง DataFrame เพื่อตรวจสอบค่าติดลบสำหรับแต่ละเขตและยอดรวม

# รวมคอลัมน์ของเขตและคอลัมน์ 'รวม'
columns_to_check = list(DIST_COLS) + ['รวม']

# ตรวจสอบว่ามีค่าติดลบหรือไม่
negative_check_results = []
for col in columns_to_check:
    has_negative = check_if_negative_exists(df, col)
    negative_check_results.append({'เขต': col, 'มีค่า < 0': has_negative})

# สร้าง DataFrame จากผลลัพธ์
df_negative_check = pd.DataFrame(negative_check_results)

# แสดงผล DataFrame
display(df_negative_check)

# นับจำนวนคอลัมน์ที่มีค่าติดลบ
num_negative_cols = df_negative_check['มีค่า < 0'].sum()
print(f'\n\nจำนวนเขตที่มีค่าติดลบ: {num_negative_cols} เขต')


,เขต,มีค่า < 0
0,วัดเลียบ,False
1,คลองเตย,False
2,ยานนาวา,False
3,บางกะปิ,False
4,มีนบุรี,False
5,สมุทรปราการ,False
6,บางพลี,False
7,สามเสน,False
8,นนทบุรี,False
9,บางใหญ่,False




จำนวนเขตที่มีค่าติดลบ: 0 เขต


ไม่พบว่าจุดข้อมูลใดมีค่าต่ำกว่า 0 หรือติดลบ นั่นคือข้อมูลได้รับการเก็บรวบรวมมาอย่างถูกต้องและมีความน่าเชื่อถือ

<br>
<br>

### Prepare 2.6 การสร้างดัชนีเวลา (Time Index): สร้างคอลัมน์ Datetime และแปลง พ.ศ. เป็น ค.ศ. เพราะปัจจุบันต้องใช้ 2 คอลัมน์ระบุเวลา -> ทำให้เหลือ 1 คอลัมน์

ปัจจุบันการจะระบุเวลาต้องใช้ column เดือนและปี ทำให้เป็นข้อมูล time series มาตรฐานโดยการนำเดือนและปีมาแปลงเป็นตัวเลข แล้วนำไปแปลงเป็น format datetime

In [ ]:
# กำหนดการ Mapping เพื่อแปลงชือ่เดือนเป็นตัวเลข
MONTH_MAP = {'ม.ค.':1, 'ก.พ.':2, 'มี.ค.':3, 'เม.ย.':4, 'พ.ค.':5, 'มิ.ย.':6,
             'ก.ค.':7, 'ส.ค.':8, 'ก.ย.':9, 'ต.ค.':10, 'พ.ย.':11, 'ธ.ค.':12}

#สลับตำแหน่ง key กับ value
MONTH_TH = {v: k for k, v in MONTH_MAP.items()}

df['เดือน_num'] = df['เดือน'].map(MONTH_MAP)

# ผลการดำเนินการ
print(f"การแมพข้อมูล {len(df)} แถว พบค่าที่ Map ไม่ได้ (NaN): {df['เดือน_num'].isna().sum()} แถว")

# ตรวจสอบ Unique อีกครั้ง (หลังแก้ไขปัญหา whitespace และ Map แล้ว)
print("Unique ของ col 'เดือน' หลังแก้ whitespace และ map ตัวเลข:", sorted(df['เดือน_num'].unique().tolist()))

การแมพข้อมูล 83 แถว พบค่าที่ Map ไม่ได้ (NaN): 0 แถว
Unique ของ col 'เดือน' หลังแก้ whitespace และ map ตัวเลข: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [ ]:
# สร้างคอลัมน์ date
# ปัญหา: pandas รองรับ timestamp เฉพาะ ปี 1677-2262
# วิธีแก้: แปลง พ.ศ. ให้เป็น ค.ศ. (ลบ 543) ก่อน แล้วค่อยสร้าง datetime

print('=== ตรวจสอบก่อนสร้าง date ===')
print(f'ปี พ.ศ. (sample): {df["ปี"].head(3).tolist()}')
print(f'เดือน_num (sample): {df["เดือน_num"].head(3).tolist()}')
print(f'ปี ค.ศ. ที่คาดหวัง: {(df["ปี"].head(3) - 543).tolist()}')

# สร้าง date
df['ปี_CE'] = df['ปี'] - 543
df['date']  = pd.to_datetime(
    df['ปี_CE'].astype(str) + '-' +
    df['เดือน_num'].astype(str).str.zfill(2) + '-01'
)

# ตรวจสอบผลลัพธ์
print()
print('=== ผลหลังสร้าง date ===')
print(f'dtype        : {df["date"].dtype}')
print(f'NaN          : {df["date"].isnull().sum()}')
print(f'min          : {df["date"].min().date()}')
print(f'max          : {df["date"].max().date()}')
display(df[['ปี','ปี_CE','เดือน','เดือน_num','date']].head(24))

=== ตรวจสอบก่อนสร้าง date ===
ปี พ.ศ. (sample): [2562, 2562, 2562]
เดือน_num (sample): [1, 2, 3]
ปี ค.ศ. ที่คาดหวัง: [2019, 2019, 2019]

=== ผลหลังสร้าง date ===
dtype        : datetime64[ns]
NaN          : 0
min          : 2019-01-01
max          : 2025-11-01


,ปี,ปี_CE,เดือน,เดือน_num,date
0,2562,2019,ม.ค.,1,2019-01-01
1,2562,2019,ก.พ.,2,2019-02-01
2,2562,2019,มี.ค.,3,2019-03-01
3,2562,2019,เม.ย.,4,2019-04-01
4,2562,2019,พ.ค.,5,2019-05-01
5,2562,2019,มิ.ย.,6,2019-06-01
6,2562,2019,ก.ค.,7,2019-07-01
7,2562,2019,ส.ค.,8,2019-08-01
8,2562,2019,ก.ย.,9,2019-09-01
9,2562,2019,ต.ค.,10,2019-10-01


สร้างคอลัมน์ date ที่เป็น format datetime เรียบร้อย สามารถใช้คอลัมน์นี้เป็น time index แทนการใช้ทั้งคอลัมน์ปีและเดือนแต่จะไม่ลบคอลัมน์เก่าทิ้งเพราะอาจมีความจำเป็นต่อการวิเคราะห์

<br>
<br>

In [ ]:
df['เดือน'].unique()

array(['ม.ค.', 'ก.พ.', 'มี.ค.', 'เม.ย.', 'พ.ค.', 'มิ.ย.', 'ก.ค.', 'ส.ค.',
       'ก.ย.', 'ต.ค.', 'พ.ย.', 'ธ.ค.'], dtype=object)

### Prepare 2.7 **สรุปผลการเตรียมข้อมูล (Preparation Log)**

In [ ]:
# 1. เตรียมรายการข้อมูลสำหรับสรุป
summary_list = []

# 2. วนลูปเช็คทีละคอลัมน์จากข้อมูลปัจจุบัน (df)
for col in df.columns:
    # ตั้งชื่อคอลัมน์เดิมเพื่อไปดึงข้อมูลเก่า (ถ้าเป็น 'รวม' ให้ไปดึงจาก '(หักไฟทำการ)')
    old_col_name = '(หักไฟทำการ)' if col == 'รวม' else col

    # เช็คว่ามีข้อมูลเก่าใน info_df หรือไม่
    has_before = old_col_name in info_df.index

    # สร้าง dictionary เก็บข้อมูลของคอลัมน์นี้
    row = {
        'รายการ': col,
        'Type (ก่อน)': info_df.loc[old_col_name, 'Type'] if has_before else '-',
        'Type (หลัง)': df[col].dtype,
        'Missing (ก่อน)': info_df.loc[old_col_name, 'Missing'] if has_before else '-',
        'Missing (หลัง)': df[col].isnull().sum(),
        'Unique (ก่อน)': info_df.loc[old_col_name, 'Unique'] if has_before else '-',
        'Unique (หลัง)': df[col].nunique(),
        'Duplicates (ก่อน)': info_df.loc[old_col_name, 'Duplicates'] if has_before else '-',
        'Duplicates (หลัง)': df.duplicated(subset=[col]).sum() if col in df.columns else '-',
        'ค่าติดลบ?': 'No' if (df[col].dtype != 'object' and df[col].dtype != 'datetime64[ns]') and (df[col] >= 0).all() else '-'
    }
    summary_list.append(row)

# 3. แปลงเป็น DataFrame และแสดงผล
summary_df = pd.DataFrame(summary_list)
display(summary_df)

,รายการ,Type (ก่อน),Type (หลัง),Missing (ก่อน),Missing (หลัง),Unique (ก่อน),Unique (หลัง),Duplicates (ก่อน),Duplicates (หลัง),ค่าติดลบ?
0,ปี,float64,int64,76,0,7,7,0,76,No
1,เดือน,object,object,0,0,16,12,0,71,-
2,วัดเลียบ,float64,float64,0,0,82,82,0,1,No
3,คลองเตย,float64,float64,0,0,82,82,0,1,No
4,ยานนาวา,float64,float64,0,0,82,82,0,1,No
5,บางกะปิ,float64,float64,0,0,83,83,0,0,No
6,มีนบุรี,float64,float64,0,0,83,83,0,0,No
7,สมุทรปราการ,float64,float64,0,0,82,82,0,1,No
8,บางพลี,float64,float64,0,0,83,83,0,0,No
9,สามเสน,float64,float64,0,0,83,83,0,0,No


หลังจากดำเนินการเตรียมข้อมูลโดยการ

- ตรวจสอบ ลบคอลัมน์ `(หักไฟทำการ)` แล้วปรับปรุงเป็นคอลัมน์ `รวม`
- เติมค่าว่าง
- แก้ไขปัญหา Inconsistent สำหรับเดือน
- สร้าง Time Index โดยใช้ตัวแปรที่เป็น datetime แทนการใช้ 2 column เพื่อเป็น key
<br>

พบว่าข้อมูลไม่มีปัญหาใดๆ มีความสอดคล้อง และเหมาะสมกับการนำไปใช้วิเคระาห์ต่อไป

<br>
<br>
<br>



## <u>EDA:</u> สำรวจลักษณะของข้อมูล (Exploratory Data Analysis - EDA)

หลังจากเตรียมข้อมูลเรียบร้อยพร้อมทำการวิเคราะห์ ทำการตรวจสอบลักษณะข้อมูลด้วยการ EDA

### สำรวจเบื้องต้นโดยใช้สถิติพรรณาพื้นฐาน

In [ ]:
# เก็บตัวเลขปี
N_YEARS_TOTAL = df['ปี'].nunique()

# ค่าเฉลี่ยรายปีของยอดรวม
yearly_total = df.groupby('ปี')['รวม'].sum()

# หาเขตที่ใช้ไฟเฉลี่ย มากที่สุด/น้อยที่สุด
avg_by_dist = df[DIST_COLS].mean().sort_values(ascending=False)
top_dist_name, top_dist_val = avg_by_dist.index[0], avg_by_dist.iloc[0]
low_dist_name, low_dist_val = avg_by_dist.index[-1], avg_by_dist.iloc[-1]

# หาเดือนที่ใช้ไฟเฉลี่ย สูงสุด/ต่ำสุด พร้อมค่า
monthly_avg = df.groupby('เดือน_num')['รวม'].mean()
peak_month_name = MONTH_TH[monthly_avg.idxmax()]
peak_month_val  = monthly_avg.max()
low_month_name  = MONTH_TH[monthly_avg.idxmin()]
low_month_val   = monthly_avg.min()

# เก็บข้อมูลในรูป Dict

EDA = {
    'ช่วงเวลา': f'{N_YEARS_TOTAL} ปี)',
    'ยอดรวมทั้งหมด (ล้านหน่วย)': f'{df["รวม"].sum():,.0f}',
    'เฉลี่ยต่อเดือน (ล้านหน่วย)': f'{df["รวม"].mean():,.1f}',
    'เดือนที่ใช้ไฟสูงสุด (เฉลี่ย)': f'{peak_month_name} ({peak_month_val:,.1f})',
    'เดือนที่ใช้ไฟต่ำสุด (เฉลี่ย)': f'{low_month_name} ({low_month_val:,.1f})',
    'เขตที่ใช้ไฟมากที่สุด (เฉลี่ย)': f'{top_dist_name} ({top_dist_val:,.1f})',
    'เขตที่ใช้ไฟน้อยที่สุด (เฉลี่ย)': f'{low_dist_name} ({low_dist_val:,.1f})',
}

# Print Output
print('=' * 45)
print('   EDA SUMMARY (หน่วย: ล้านหน่วย)')
print('=' * 45)
for k, v in EDA.items():
    print(f'  {k:<35}: {v}')

   EDA SUMMARY (หน่วย: ล้านหน่วย)
  ช่วงเวลา                           : 7 ปี)
  ยอดรวมทั้งหมด (ล้านหน่วย)          : 365,797
  เฉลี่ยต่อเดือน (ล้านหน่วย)         : 4,407.2
  เดือนที่ใช้ไฟสูงสุด (เฉลี่ย)       : พ.ค. (4,835.4)
  เดือนที่ใช้ไฟต่ำสุด (เฉลี่ย)       : ธ.ค. (3,947.2)
  เขตที่ใช้ไฟมากที่สุด (เฉลี่ย)      : บางพลี (439.2)
  เขตที่ใช้ไฟน้อยที่สุด (เฉลี่ย)     : วัดเลียบ (89.5)


- สาเหตุที่เดือน พ.ค. ใช้ไฟสูงที่สุดอาจจะมาจากอุณหภูมิที่สูงขึ้น และเดือน ธ.ค. ใช้ไฟต่ำที่สุดมาจากการที่ประเทศไทยเป็นช่วงฤดูหนาว ถึงแม้ว่าเดือนพ.ค.จะเป็นช่วง Low season และ ธ.ค.จะเป็น high season ก็ตาม
- สาเหตุที่เขตบางพลีใช้ไฟเยอะเนื่องจากอยู่ในจังหวัดสมุทรปราการซึ่งบางพีลเองก็เป็นเขตนิคมอุตสาหกรรมที่มีจำนวนโรงงานมากที่สุด ประมาณ400 โรงงาน (นับเฉพาะใน 18 เขตนี้)
  - Source จำนวนโรงงาน:
  https://userdb.diw.go.th/factory/ieat.asp#:~:text=Table_title:%20ข้อมูลโรงงานในเขตการนิคมแห่งประเทศไทย%20Table_content:%20header:%20%7C%20รหัส%20%7C,(ต.บ้านกลาง%20อ.เมือง%20ลำพูน)%20%7C%20จำนวน:%20140%20%7C

<br>  

- ส่วนสาเหตุที่วัดเลียบมียอดใช้ไฟฟ้าน้อยอาจมาจาก พื้นที่ขนาดเล็ก จำนวนผู้ใช้ไฟฟ้าน้อยที่สุดในกรุงเทพ และเป็นย่านเมืองเก่า ไม่มีอุตสาหกรรมหนัก
  - Soruce สถิติจำนวนผู้ใช้ไฟฟ้าวัดเลียบ: https://opendata.mea.or.th/dataset/statistics-by-area

### **คำถามการ EDA**

EDA 1: ลักษณะข้อมูลแต่ละเขต <br>
EDA 2: กราฟเส้นการใช้ไฟฟ้ารวมรายปี <br>
EDA 3: การใช้ไฟฟ้าแต่ละเขตมีความสัมพันธ์เชิงเส้นเกิดขึ้นไหม <br>

### คำถามอื่นๆที่เป็นการวิเคราะห์เชิงสำรวจเช่นกัน ได้แก่

- แนวโน้ม, Seasonal ของยอดรวมทุกเขต --> คำถามข้อที่ 1
- ปริมาณการใช้ไฟฟ้ารวม เปรียบเทียบตามเขต --> คำถามข้อที่ 2
- การเติบโตของการใช้ไฟฟ้าตามเขต --> คำถามข้อที่ 2
- Seasonality ของแต่ละเขต --> คำถามข้อที่ 3

### *<u>จะไม่ถูกนำมาแสดงในการ EDA เนื่องจากเนื้อหามีความทับซ้อนกับส่วนคำถามหลัก 3 ข้อจาก Thackle</u>*

### EDA 1: ตรวจสอบข้อมูลรายเขตด้วย **Dashboard**

In [ ]:
# EDA: ตรวจสอบลักษณะของข้อมูลอนุกรมโดยการสร้างแดชบอร์ดสำหรับตรวจสอบลักษณะของข้อมูลรายเขต และยอดรวมทุกเขต

# ดึงรายชื่อเขตที่เก็บไว้ใน DIST_COLS
districts_list = list(DIST_COLS)
districts_list.append('รวม')
fig = go.Figure()

# สร้างเส้นกราฟสำหรับทุกเขตในแดชบอร์ด
for i, dist in enumerate(districts_list):
    fig.add_trace(
        go.Scatter(x=df['date'], y=df[dist], mode='lines+markers',
                   name=dist, visible=(i==0))
    )

# สร้างปุ่มตัวกรอง (Dropdown) สำหรับเลือกเขตที่ต้องการ
buttons = []
for i, dist in enumerate(districts_list):
    visible_array = [False] * len(districts_list)
    visible_array[i] = True
    buttons.append(
        dict(label=dist,
             method="update",
             args=[{"visible": visible_array},
                   {"title": f"<b>EDA Dashboard การใช้ไฟฟ้า - {dist}</b>"}])
    )
# จัด layout ของ dashboard และกำหนดให้หน้า default เป็นหน้าที่ 18 ก็คือหน้า 'รวม'
fig.update_layout(
    updatemenus=[dict(
        active=18, buttons=buttons,
        x=0.0, xanchor="left", y=1.2, yanchor="top",
        direction="down", showactive=True,
    )],
    font=dict(family="TH Sarabun New", size=24),
    title_font_size=28,
    title=f"<b>การใช้ไฟฟ้าในกรุงเทพและปริมณฑล 18 พื้นที่ - {districts_list[0]}</b>",
    xaxis_title="<b>เดือน/ปี</b>",
    yaxis_title="<b>การใช้พลังงานไฟฟ้า</b>",
    xaxis=dict(tickformat="%m/%y", dtick="M2"),
    margin=dict(t=140),
    showlegend=False
)

fig.show()

จากการตรวจสอบทั้ง 18 เขต
- พบว่าหลายๆเขตระบุแนวโน้มระยะยาวตลอดทุกช่วงได้ค่อนข้างยาก สามารถระบุได้เพียง เขตบางพลี เขตบางบัวทอง ที่เห็นได้ชัดว่ามีแนวโน้มเพิ่มขึ้น รวมถึงเขตราษฎร์บูรณะ ที่สามารถระบุได้ชัดเจนว่ามีแนวโน้มลดลงเช่นกัน
- เขตลาดกระบังมีการใช้ไฟฟ้าอย่างผิดปกติมากในเดือน 9 ปี 2020

**ผลลัพธ์** :
- ควรมีการใช้ KPI ในการวัดการเติบโตมากกว่าการมองกราฟด้วยสาตาซึ่งอาจทำให้ระบุการเติบโตได้ดียิ่งขึ้นโดย metric ที่แนะนำคือ อัตราการเติบโตเฉลี่ยต่อปี (CAGR) ที่สามารถบอกได้ว่าแต่ละปีจะโตขึ้นทบต้นเป็นกี่ % แล้วนำมาเปรียบเทียบกัน <br>
- ในเดือนกันยายนปี 2020 ได้เกิดเหตุการณ์สำคัญที่อาจมีส่วนทำให้ยอดการใช้ไฟของเขตลาดกระบังสูงขึ้นอย่างผิดปกติดังนี้
  - ฝนหนักถลุ่มกรุงเทพ ด้วยสภาวะที่พื้นที่ลาดกระบังระบายน้ำได้ยากเนื่องจากเป็นแอ่ง ประกอบกับเป็นสถานนิคมอุตสาหกรรม ทำให้ต้องมีการใช้ไฟฟ้าในการสูบหน้าเพื่อรักษาเสถียรภาพของการผลิต ส่งผลให้ต้องใช้ไฟมากขึ้น อย่างไรก็ตามเนื่องจากไม่สามารถหาข่าวที่ระบุถึงความเสียหายในเขตลาดกระบัง และไม่ปรากฎข่าวอื่นๆที่มีความเกี่ยวข้องกับปริมาณการใช้ไฟฟ้า จึงไม่สามารถระบุชี้ชัดได้ว่า สาเหตุมาจากฝนหนักหรือมาจากสาเหตุใดกันแน่ <br>
  source (ข่าวฝนหนักกรุงเทพ): https://www.prachachat.net/breaking-news/news-526464

### EDA 2: การใช้ไฟฟ้ารวมแยกรายปี <br>

In [ ]:
# สร้าง DataFrame สำหรับพล็อตกราฟรายปี
df_yearly = yearly_total.reset_index()
df_yearly.columns = ['ปี', 'ยอดรวม']

# สร้างกราฟเส้นด้วย
fig = px.line(df_yearly, x='ปี', y='ยอดรวม',
              markers=True,
              text=df_yearly['ยอดรวม'].apply(lambda x: f'{x:,.0f}'),
              title='<b>EDA 2: แนวโน้มการใช้ไฟฟ้ารวมรายปี (2562 - 2568)</b>')

# ปรับแต่งหน้าตาของกราฟ
fig.update_traces(line=dict(color='#1E88E5', width=3),
                  marker=dict(size=12, color='#FB8C00'),
                  textposition='top center')

fig.update_layout(
    xaxis_title='<b>ปี (พ.ศ.)</b>',
    yaxis_title='<b>ยอดรวมการใช้ไฟฟ้า (ล้านหน่วย)</b>',
    font=dict(family='TH Sarabun New', size=20),
    title_font_size=26,
    margin=dict(t=100),
    xaxis=dict(dtick=1)
)

fig.show()

- ปริมาณการใช้ไฟฟ้าปี 62-666 มีลักษณะลดลงแล้วฟื้นกลับมาจุดเดิม ในปี 2567 ดีดตัวสูงขึ้นอีกคนั้ง ส่วนปร 2568 ลดระดับลงไป
- ในปี 2562-2567ที่เห็นว่ากราฟมีลักษณะลดลงแล้วฟื้่นกลับมามาจากการถดถอยของภาวะการท่องเที่ยวและเศรษฐกิจจากภาวะโควิด 19  จะเห็นได้ว่าปีที่การท่องเที่ยวและเศรษฐกิจเริ่มฟื้น ปริมาณการใช้ไฟฟ้าจะเพิ่มขึ้นด้วย <br> <br> ***นอกจานี้อุณหภูมิที่สูงขึ้นก็ทำให้ยอดการใช้ไฟฟ้าเพิ่มขึ้นด้วย*** <br> <br>
  Source : https://www.bangkokbiznews.com/business/economic/1122015
  - พบว่าในปี 2567 และมีปรากฎการณ์เอลนีโญร่วมด้วยซึ่งทำให้อุณหภูมิสูงขึ้น
    - Source: https://www.enervision.co.th/news/energynews67-04-30-01/#:~:text=เดือน%20เม.ย.%202567%20มี,8%20ปี%20%7C%20Energy%20News%**20Center**
- ในปี 2568 สาเหตุที่ยอดการใช้ไฟฟ้าลดลงเป็นเพราะว่าข้อมูลยังต้องรออีก 1 เดือน (ข้อมูลมีถึงแค่เดือน พ.ย. คาดว่าเมื่อรวมกับข้อมูลเดือนธันวาคม (คาดการณ์ว่าประมาณ 4000 ล้านหน่วย) ยอดการใช้ไฟฟ้าจะใกล้เคียงกับปี 2567

**Key:** จาก bullet ที่ 2 ด้านบนจะเห็นว่าแหล่งข่าวบอกว่า ***<u>อุณหภูมิมีความสัมพันธ์กับปริมาณการใช้ไฟฟ้า</u>*** จะทำการหาคำตอบในประเด็นนี้ว่าจริงหรือไม่ในหัวข้อ **คำถามส่วนตัว 4.3*- Correlation Analysis* ส่วนปัจจัยอื่นๆที่เนื้อข่าวบอกเช่นเศรษฐกิจ และ จำนวนนักท่องเที่ยว ได้มีงานวิจัยศึกษาไปแล้วว่ามีผลจริง

### EDA 3: การใช้ไฟฟ้าแต่ละเขตมีความสัมพันธ์เชิงเส้นเกิดขึ้นไหม <br>

In [ ]:
# 1. คำนวณค่าสหสัมพันธ์ (Correlation Matrix) ระหว่างเขต
corr_matrix = df[DIST_COLS].corr()

# 2. พล็อต Heatmap ด้วย Plotly Express
fig = px.imshow(corr_matrix,
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                text_auto='.2f',
                aspect='auto',
                title='<b>EDA 3: ความสัมพันธ์เชิงเส้นของการใช้ไฟฟ้าในแต่ละเขต (Correlation Heatmap)</b>')

# 3. ปรับแต่งหน้าตาของกราฟ
fig.update_traces(textfont=dict(size=14))
fig.update_layout(
    margin=dict(t=100, b=80),
    xaxis_title='<b>เขต</b>',
    yaxis_title='<b>เขต</b>',
    xaxis=dict(tickangle=-45),
    font=dict(family='TH Sarabun New', size=20),
    title_font_size=24
)

fig.show()

- จะเห็นว่าโดยส่วนใหญ่และจะมีความสัมพันธ์ตั้งแต่ระดับต่ำไปจนถึงสูง
- มีความสัมพันธ์กันทางเดียวกันทุกๆเขตไม่มีเขตไหนสวนทางกัน
- หมายความว่าอาจจะมีแพทเทิร์น Seasonality ที่ทุกเขตได้รับอิทธิพลเหมือนๆกันในช่วงเวลาเดียวกัน แต่ขนาดอิทธิพลอาจจะต่างกัน ควรหาอิทธิพลของฤดูกาลแยกตามแต่ละเขต (ทำการวิเคราะห์ส่วนนีเพิ่มเติมในคำถามที่ 3)

## คำถาม Thackle

## <u>1. คำถามที่ 1</u> - การใช้ไฟฟ้ารวมเปลี่ยนแปลงอย่างไรใน 7 ปี Trend & Anomaly Detection
**คำถามข้อที่ 1.** การใช้ไฟฟ้ารวมเปลี่ยนแปลงอย่างไรใน 7 ปี? (Trend & Anomaly Detection)
- **โจทย์**: สร้างกราฟแสดงแนวโน้ม (Trend), รูปแบบฤดูกาล (Seasonality) และตรวจจับจุดผิดปกติ (Anomaly) จากข้อมูลหน่วยจำหน่ายไฟฟ้ารวมรายเดือนปี 2562–2568
- **สิ่งที่ต้องหาคำตอบ**: ระบุเหตุการณ์ที่อธิบายจุดผิดปกติ (เช่น COVID-19, คลื่นความร้อน, มาตรการรัฐ) และวิเคราะห์แนวโน้มการใช้ไฟฟ้าในปี 2568 สื่อสารทิศทางการใช้ไฟฟ้าว่าสะท้อนภาวะเศรษฐกิจเมืองอย่างไร
- **เครื่องมือวิเคราะห์ (KPI / Method)**: การวิเคราะห์แนวโน้มเคลื่อนที่เฉลี่ย (Rolling Mean), ดัชนีความผันผวนของฤดูกาล (Seasonal Index) และวิธีตรวจจับค่าผิดปกติด้วยหลักการควบคุม (Bollinger Band )

<br>
<br>
<br>

<table style="width:100%; font-size:16px">
  <tr>
    <th>Visualization</th>
    <th>ประเภทกราฟ</th>
    <th>เป้าหมาย</th>
  </tr>
  <tr>
    <td>Visualization 1.1</td>
    <td>Line + Rolling Mean + Bollinger Band</td>
    <td>แนวโน้มรายเดือนและจุดผิดปกติ</td>
  </tr>
  <tr>
    <td>Visualization 1.2</td>
    <td>Box Plot รายเดือน</td>
    <td>อิทธิพลของฤดูกาล</td>
  </tr>
</table>

<br>
<br>
<br>


### Visualization 1.1 - แนวโน้มรายเดือนและการตรวจจับ Anomaly

**เหตุผลที่เลือก Moving Average + Standard Dev.:**
เนื่องจาก Moving Average ช่วยให้ข้อมูลเรียบ ทำให้เห็นการเคลื่อนไหวแนวโน้มของข้อมูลชัดเจน นอกจากนี้เมื่อใช้ BollingGer Band (หรือ k คูณด้วย rolling STD.) จะทำให้ตรวจจับค่าที่อยู่นอกช่วง k * ส่วนเบี่ยงเบนมาตรฐานในอดีต โดยการเพิ่มขนาด k จะทำให้การคัดกรองจุดผิดปกติ มีความอลุ่มอล่วยต่อการเป็นค่าผิดปกติมากขึ้น)

**Source:** https://docs.datarobot.com/en/docs/modeling/special-workflows/unsupervised/anomaly-detection.html

In [ ]:
# Visualization 1.1 แนวโน้ม + เส้น Moving Average + เส้น Bollinger Band(k x Std.)

ts = df.set_index('date')['รวม'].sort_index()

ROLL_WINDOW = 12 # Moving Average 12 เดือน

roll_mean = ts.rolling(ROLL_WINDOW, center=False, min_periods=6).mean()
roll_std  = ts.rolling(ROLL_WINDOW, center=False, min_periods=6).std()
THRESH    = 2 # จุดที่ห่างจาก Moving Average เกิน 2 SD นับเป็น Anomaly
anomaly   = ts[abs(ts - roll_mean) > THRESH * roll_std]

print(f'จำนวน Anomaly ที่พบ: {len(anomaly)} จุด')

จำนวน Anomaly ที่พบ: 3 จุด


In [ ]:
# พล็อต Visualization 1.1
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.index, y=(roll_mean - THRESH*roll_std).fillna(method='bfill').fillna(method='ffill'),
                         mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=ts.index, y=(roll_mean + THRESH*roll_std).fillna(method='bfill').fillna(method='ffill'),
                         mode='lines', line=dict(width=0), fill='tonexty', fillcolor='#c6d3e3',
                         name=f'ช่วงปกติ (±{THRESH} SD)'))
fig.add_trace(go.Scatter(x=ts.index, y=ts.values, mode='lines', name='ยอดรวมรายเดือน', line=dict(color='#1E88E5', width=1.5)))
fig.add_trace(go.Scatter(x=roll_mean.index, y=roll_mean.values, mode='lines', name=f'Trend (Rolling {ROLL_WINDOW} เดือน)',
                         line=dict(color='#FB8C00', width=2.5, dash='dash')))
fig.add_trace(go.Scatter(x=anomaly.index, y=anomaly.values, mode='markers', name=f'Anomaly (> {THRESH} SD)',
                         marker=dict(color='#E53935', size=10)))

for yr in df['date'].dt.year.unique():
    import pandas as pd
    fig.add_vline(x=pd.Timestamp(f'{yr}-01-01').timestamp() * 1000, line_width=1, line_dash="dot", line_color="grey")
    fig.add_annotation(x=pd.Timestamp(f'{yr}-06-01').timestamp() * 1000, y=ts.max()*1.05, text=str(yr+543),
                       showarrow=False, font=dict(size=20, color='grey'))

fig.update_layout(title='<b>Visualization 1.1 - แนวโน้มการใช้ไฟฟ้ารวมรายเดือน<br>พร้อม Anomaly Detection (18 เขต MEA ปี 2562-2568)</b>',
                  xaxis_title='<b>เดือน</b>', yaxis_title='<b>ยอดรวม (ล้านหน่วย)</b>',
                  margin=dict(t=150),
                  font=dict(family='TH Sarabun New', size=20),
                  title_font_size=24, hovermode='x unified',
                  legend=dict(yanchor="bottom", y=1.02, xanchor="right", x=1.0, orientation="h"))
fig.show()

### **<u>ตอบคำถาม</u>**
- จากการวิเคราะห์แนวโน้ม ข้อมูลมีลักษณะค่อนข้างกลับเข้าหาค่าเฉลี่ย (มีลักษณะทรงตัว)
- ข้อมูลมีความแปรปรวนสูงขึ้น overtime
- มีจุด anomaly 3 จุด ที่อยู่นอกช่วง 2*SD (ประมาณ 95% ของข้อมูลในอดีต)
  - จุดแรก **Jan 2021** มีลักษณะเป็นจุดตกต่ำผิดปกติ สาเหตุมาจากโควิด-19 ที่ดึงเศรษฐกิจไทยลงมา ทำให้ประเทศไทยบริโภคไฟฟ้าน้อยลง นอกจานี้ยังเป็นช่วงฤดูหนาวอาจเป็นปัจจัยเสริมแรงฉุดการใช้ไฟฟ้าไปอีก (ได้ทำการทดสอบว่าอุณหภูมิมีผลในหัวข้อคำถามส่วนตัว 4.3)
  - จุดที่ 2 **May 2023** ลักษณะเป็นจุด peak ผิดปกติเป็นช่วงที่ประเทศไทยกำลังฟื้นเศรษฐกิจจากโควิด
  ดูเพิ่มเติมเรื่องโควิดที่หัวข้อ EDA 2 และด้วยสภาพอากาศที่เป็นช่วงฤดูร้อน และปรากฏการณ์เอลนีโญ ที่เสริมแรงต่อ ทำให้เกิดการใช้ไฟฟ้าพุ่งแรงมากๆ
    - Source เรื่องเอลนีโญ :
      https://www.salika.co/2023/07/15/el-nino-phenomenon-thai-economic/#:~:text=จากบทวิเคราะห์ล่าสุด%20ศูนย์วิจัย,พฤษภาคม%202566%20ที่ผ่านมา
  - จุดที่ 3 **Jan 2025** หากมองจริงๆแล้วอาจจะไม่ได้เป็นจุดที่มีความผิดปกติมากเพียงแต่ในปี 2024 ที่ประเทศฟื้นเศรษฐกิจอย่างมากหลังจากช่วงโควิด ได้หนุนการใช้ไฟฟ้าจนจุดต่ำสุดในปีไม่ตกกลับมาที่ช่วงปกติที่ควรจะเป็น และเมื่อเวลาผ่านไปก็กลับสู่สภาวะปกติในปี 2025 นั่นคือในปี 2025 ช่วงมกราคม ค่ากลับมาใกล้เคียงกับปีก่อนๆ (สรุป ค่าปี 2025 ผิดปกติจากรอบปีก่อนหน้าคือปี 2024 แต่จริงๆแล้วมันยังปกติเมื่อเทีบกับปี 2019-2023 เป็นเพราะว่า 2024 ประเทศไทยโตจนมีการใช้ไฟฟ้าเยอะมากๆทั้งปีจากการฟื้นฟูเศรษฐกิจจากโควิด)

### Visualization 1.2 - รูปแบบฤดูกาล (Seasonality) รายเดือน

**เหตุผลที่เลือก Box Plot รายเดือน:**
Box Plot แต่ละเดือนสรุปการกระจายตัวของยอดใช้ไฟในทุกปีที่ผ่านมา
ทำให้เห็น Peak และ Low Season ได้ทันที
ความกว้างของ Box สะท้อนการกระจายของการใช้ไฟในเดือนนั้น


**ดัชนีฤดูกาล (Seasonal Index)**:
$$ \text{Seasonal Index}_{(\%)} = \left( \frac{\text{Average for the Season}}{\text{Average of all data}} \right) \times 100 $$


In [ ]:
# Visualization 1.2 แดชบอร์ด Box Plot รูปแบบฤดูกาลรายเดือน และ Seasonal Index

month_labels = ['ม.ค.','ก.พ.','มี.ค.','เม.ย.','พ.ค.','มิ.ย.','ก.ค.','ส.ค.','ก.ย.','ต.ค.','พ.ย.','ธ.ค.']

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

tmp_df = df.copy()
tmp_df['เดือน_cat'] = pd.Categorical(tmp_df['เดือน'], categories=month_labels, ordered=True)
tmp_df = tmp_df.sort_values('เดือน_cat')

# คำนวณ Seasonal Index
season_avg = tmp_df.groupby('เดือน_num')['รวม'].mean().reset_index()
overall_avg = tmp_df['รวม'].mean()
season_avg['SI'] = (season_avg['รวม'] / overall_avg) * 100

# Median รายเดือน
monthly_mn = tmp_df.groupby('เดือน_num')['รวม'].median().reset_index()

print('คำนวณข้อมูล Box Plot เรียบร้อย')

คำนวณข้อมูล Box Plot เรียบร้อย


In [ ]:
# พล็อต Visualization 1.2 (Dashboard)

fig = make_subplots(specs=[[{"secondary_y": True}]])

# ออกแบบการแสดง boxplot บนกราฟ
fig_box = px.box(tmp_df, x='เดือน_cat', y='รวม', points='outliers', color_discrete_sequence=['#1E88E5'])
for trace in fig_box.data:
    trace.name = 'การกระจายข้อมูลรายเดือน'
    trace.showlegend = False
    fig.add_trace(trace, secondary_y=False)

# ออกแบบการแสดง Median
fig.add_trace(go.Scatter(x=[month_labels[i-1] for i in monthly_mn['เดือน_num']],
                         y=monthly_mn['รวม'], mode='lines+markers', name='Median (แกนหลักซ้าย)',
                         marker=dict(symbol='diamond', size=10, color='#FB8C00')),
              secondary_y=False)

# ออกแบบการแสดง Seasonal Index
fig.add_trace(go.Scatter(x=[month_labels[i-1] for i in season_avg['เดือน_num']],
                         y=season_avg['SI'], mode='lines+markers', name='Seasonal Index % (แกนรองขวา)',
                         marker=dict(symbol='circle', size=10, color='#E53935'), visible=False),
              secondary_y=True)

# ปุ่ม Dashboard ใช้สลับกราฟ
buttons = [
    dict(label="Box Plot + Median",
         method="update",
         args=[{"visible": [True, True, False]},
               {"title": "<b>Visualization 1.2 - รูปแบบฤดูกาลการใช้ไฟฟ้ารวม (Box Plot & Median)</b>"}]),
    dict(label="Seasonal Index",
         method="update",
         args=[{"visible": [False, False, True]},
               {"title": "<b>Visualization 1.2 - รูปแบบฤดูกาลการใช้ไฟฟ้ารวม (Seasonal Index)</b>"}])
]

fig.update_layout(
    updatemenus=[dict(
        active=0, buttons=buttons,
        x=0.0, xanchor="left", y=1.15, yanchor="top",
        direction="down", showactive=True,
    )],
    title='<b>Visualization 1.2 - รูปแบบฤดูกาลการใช้ไฟฟ้ารวม (Box Plot & Median)</b>',
    margin=dict(t=150),
    font=dict(family='TH Sarabun New', size=20), title_font_size=24
)

fig.update_xaxes(title_text="<b>เดือน</b>")
fig.update_yaxes(title_text="<b>ยอดรวม (ล้านหน่วย)</b>", secondary_y=False)
fig.update_yaxes(title_text="<b>Seasonal Index (%)</b>", secondary_y=True)
fig.show()


### **<u>ตอบคำถาม</u>**
- ปริมาณการใช้ไฟฟ้ารวมของทั้ง 18 เขตมีฤดูกาลในรอบปี คือ มีจุดพีคคือทุกเดือนพฤษภาคม จุดตกต่ำที่สุดคือเดือนธันวาคม โดยระหว่างเดือนมกราคมถึงเมษายนจะมีปริมาณการใช้ไฟฟ้าไต่ระดับลง และเริ่มพลิกกลับมาลบลงภายหลังเดือน พ.ค.

<br>

**<u>Action</u>**:

- ควรมีการ stock ไฟฟ้าสำรองให้พอสำหรับช่วงเดือนพฤษภาคมเพื่อป้องกัน spike ผิดปกติ ส่วนในช่วงเดือนธ.ค. สามารถลดปริมาณการผลิตไฟฟ้าสำหรับเขตเหล่านี้ได้ ซึ่งอาจจะมีการนำปริมาณไฟฟ้าสำรองมาใช้เล็กน้อยเพื่อลดต้นทุนในการผลิตไฟฟ้า และช่วยลดค่า FT ของผู้บริโภค
- รัฐควรออกมาตรการค่าไฟเพื่อสำหรับประชาชนในช่วงเดือนมี.ค.-พ.ค. เนื่องจากอุณหภูมิประเทศไทยสูงขึ้นเรื่อยๆทุกปี ควรมีการลดความเหลื่อมล้ำในการเข้าถึงปริมาณพลังงาน เนื่องจากค่าไฟตอนนี้เก็บในอัตราก้าวหน้า
- สนับสนุนการใช้พลังงานหมุนเวียนในช่วงฤดูร้อนที่มีความต้องการพลังงานไฟฟ้าสูง

## <u>2. คำถามที่ 2</u> - เขตไหนใช้ไฟมากที่สุด และใครโตเร็วที่สุด District Ranking & Growth

- **โจทย์**: สร้างกราฟแสดงปริมาณการใช้ไฟฟ้าและอัตราการเติบโตของทั้ง 18 เขตจำหน่ายพร้อมกันในกราฟเดียว
- **สิ่งที่ต้องหาคำตอบ**: ระบุเขตที่ใช้ไฟมากและยังโตต่อ และเขตที่ใช้น้อยแต่โตเร็ว ระบุพื้นที่เมืองที่กำลังขยายตัว
- **เครื่องมือวิเคราะห์ (KPI / Method)**: การใช้ไฟฟ้าเฉลี่ยต่อเดือนรายเขต, สถิติอัตตราเติบโตแบบทบต้น (CAGR) และ Scatter plot

<br>
<br>

| กลุ่ม | ลักษณะ | ความหมาย |
|-------|--------|----------|
| **Q1** | ใช้มาก + โตเร็ว | เขตหลักที่ต้องลงทุนโครงสร้างพื้นฐาน |
| **Q2** | ใช้น้อย + โตเร็ว | เขตที่กำลังขยายตัว |
| **Q3** | ใช้มาก + โตช้า | เขตใหญ่ที่เติบโตอิ่มตัวแล้ว  |
| **Q4** | ใช้น้อย + โตช้า | เขตขนาดเล็กที่ไม่ค่อน progress |

<br>
<br>

| Visualization | ประเภทกราฟ | เป้าหมาย |
|--------------|-----------|----------|
| Visualization 2.1 | Scatter | จัดกลุ่ม 18 เขตตามยอดใช้ไฟและการเติบโต |

<br>

### ตอบตำถามการโตไวโดยใช้ CAGR <br>
**อัตราการเติบโตแบบทบต้น (Compound Annual Growth Rate - CAGR)**:
<br>

$$ CAGR = \left( \frac{V_{final}}{V_{begin}} \right)^{\frac{1}{n}} - 1 $$

<br>

*โดย $V_{final}$ คือยอดรวมปีล่าสุด, $V_{begin}$ คือยอดรวมปีแรกสุด, และ $n$ คือระยะเวลา (ปี)*


In [ ]:
# ผูกสูตรอัตราการเติบโตรวม (CAGR - ปี 2562 ถึงปีล่าสุด)
n_years = df['ปี'].nunique() - 1
cagr = ((yearly_total.iloc[-1] / yearly_total.iloc[0]) ** (1/n_years) - 1) * 100

In [ ]:
# ขออนุญาต comment เยอะครับเนื่องจากจำเป็นต่อผลที่ได้บนกราฟ

# คำนวณค่าเฉลี่ยการใช้ไฟฟ้าต่อเดือนของแต่ละเขตโดยหาค่าเฉลี่ยจากข้อมูลทุกแถวในคอลัมน์นั้น
avg_dist = df[DIST_COLS].mean()

# รวมยอดการใช้ไฟฟ้าแยกตามกลุ่มปีเพื่อเปรียบเทียบปริมาณการใช้ไฟรายปีของแต่ละเขต
yearly_dist = df.groupby('ปี')[DIST_COLS].sum()

# คำนวณอัตราการเติบโตสะสมโดยเทียบส่วนต่างของปีล่าสุดกับปีแรกสุดในชุดข้อมูล
growth_rate = ((yearly_dist.iloc[-1] - yearly_dist.iloc[0]) / yearly_dist.iloc[0] * 100)

# หาผลรวมการใช้ไฟฟ้าสะสมทั้งหมดของแต่ละเขตตลอดระยะเวลาเจ็ดปีที่ปรากฏในข้อมูล
total_dist = df[DIST_COLS].sum()

# ใช้ค่ามัธยฐานของค่าเฉลี่ยรายเดือนของทุกเขตมาเป็นจุดตัดบนแกนนอน
med_x = avg_dist.median()

# ใช้ค่ามัธยฐานของอัตราการเติบโตของทุกเขตมาเป็นจุดตัดบนแกนตั้ง
med_y = growth_rate.median()

import pandas as pd
import plotly.express as px

# ฟังก์ชันกำหนดสีตามตำแหน่งเชิงสถิติเปรียบเทียบกับค่ากลางของกลุ่ม
def quad_color(x, y):
    # เขตที่มีค่าเฉลี่ยสูงกว่าค่ากลางและมีอัตราเติบโตสูงกว่าค่ากลางใช้สีแดง
    if x >= med_x and y >= med_y: return '#E53935'
    # เขตที่มีค่าเฉลี่ยน้อยกว่าค่ากลางแต่มีอัตราเติบโตสูงกว่าค่ากลางใช้สีเขียว
    if x <  med_x and y >= med_y: return '#43A047'
    # เขตที่มีค่าเฉลี่ยสูงกว่าค่ากลางแต่มีอัตราเติบโตน้อยกว่าค่ากลางใช้สีส้ม
    if x >= med_x and y <  med_y: return '#FB8C00'
    # เขตที่มีค่าเฉลี่ยน้อยกว่าค่ากลางและมีอัตราเติบโตน้อยกว่าค่ากลางใช้สีน้ำเงิน
    return '#1E88E5'

# สร้างรายการสีโดยอ้างอิงตำแหน่งของแต่ละเขตว่าอยู่ตรงไหนในกราฟ
colors = [quad_color(avg_dist[d], growth_rate[d]) for d in DIST_COLS]

# รวบรวมผลลัพธ์ที่คำนวณได้ทั้งหมดมาสร้างตารางใหม่เพื่อใช้กำหนดค่าในกราฟการกระจาย Scatter polot
tmp_df = pd.DataFrame({
    'เขต': DIST_COLS,
    'avg_dist': avg_dist.values,
    'growth_rate': growth_rate.values,
    'total_dist': total_dist.values,
    'color': colors
})

print('คำนวณข้อมูลสำหรับ Plot เรียบร้อย')

คำนวณข้อมูลสำหรับ Plot เรียบร้อย


In [ ]:
# Visualization 2.1 - Scatter Plot ยอดเฉลี่ย vs Growth Rate ของ 18 เขต

# ออกแบบการพล็อต Visualization 2.1
fig = px.scatter(tmp_df, x='avg_dist', y='growth_rate', text='เขต', size='total_dist', color='color',
                 color_discrete_map='identity', hover_name='เขต')
fig.update_traces(textposition='top center', textfont_size=20)
fig.add_vline(x=med_x, line_width=1, line_dash="dash", line_color="grey")
fig.add_hline(y=med_y, line_width=1, line_dash="dash", line_color="grey")

quad_labels = [
    (0.85, 0.95, 'ใช้มาก + โตเร็ว<br>(ลงทุนด่วน)', '#E53935'),
    (0.15, 0.95, 'ใช้น้อย + โตเร็ว<br>(Emerging)', '#43A047'),
    (0.85, 0.05, 'ใช้มาก + โตช้า<br>(Mature)', '#FB8C00'),
    (0.15, 0.05, 'ใช้น้อย + โตช้า<br>(Stable)', '#1E88E5')
]
for tx, ty, label, col in quad_labels:
    fig.add_annotation(x=tx, y=ty, xref='paper', yref='paper', text=f"<b>{label}</b>",
                       showarrow=False, font=dict(size=20, color=col), bgcolor='white', opacity=0.8)

# กำหนดส่วนสูงของกราฟเนื่องจากตอนแรกผู้ศึกษาพบว่ากราฟค่อนข้างบีบอัดในแนวแกน Y
fig.update_layout(title=f'<b>Visualization 4.1 - สัดส่วนการใช้ไฟของแต่ละเขตตลอด {N_YEARS_TOTAL} ปี</b>',
                  margin=dict(t=120), height=1000,
                  xaxis_title='<b>ยอดใช้ไฟเฉลี่ยต่อเดือน (ล้านหน่วย)</b>', yaxis_title='<b>Growth Rate รวม (%) ปี 2562 → ล่าสุด</b>',
                  font=dict(family='TH Sarabun New', size=20), title_font_size=24, showlegend=False)
fig.show()

In [ ]:
# คำนวณข้อมูลสำหรับแบ่งกลุ่มใหม่เพื่อให้แน่ใจว่าตัวแปรถูกต้อง
avg_dist = df[DIST_COLS].mean()
yearly_dist = df.groupby('ปี')[DIST_COLS].sum()
growth_rate = ((yearly_dist.iloc[-1] - yearly_dist.iloc[0]) / yearly_dist.iloc[0] * 100)

# สร้าง DataFrame สำหรับกรองข้อมูล
df_ranking = pd.DataFrame({
    'เขต': DIST_COLS,
    'avg_dist': avg_dist.values,
    'growth_rate': growth_rate.values
})

med_x = df_ranking['avg_dist'].median()
med_y = df_ranking['growth_rate'].median()

# กรองรายชื่อเขตตามเงื่อนไข
high_usage_high_growth = df_ranking[(df_ranking['avg_dist'] >= med_x) & (df_ranking['growth_rate'] >= med_y)]['เขต'].tolist()
low_usage_high_growth = df_ranking[(df_ranking['avg_dist'] < med_x) & (df_ranking['growth_rate'] >= med_y)]['เขต'].tolist()

print("=== สรุปผลการวิเคราะห์แบ่งกลุ่มเขต ===")
print(f"1. เขตที่ใช้ไฟมากและยังโตต่อ): {', '.join(high_usage_high_growth)}")
print(f"2. เขตที่ใช้น้อยแต่โตเร็ว): {', '.join(low_usage_high_growth)}")

=== สรุปผลการวิเคราะห์แบ่งกลุ่มเขต ===
1. เขตที่ใช้ไฟมากและยังโตต่อ): บางกะปิ, บางพลี, บางเขน, บางนา
2. เขตที่ใช้น้อยแต่โตเร็ว): มีนบุรี, บางใหญ่, ธนบุรี, บางบัวทอง, ลาดกระบัง


### **<u>ตอบคำถาม</u>**
เขตที่กำลังใช้ไฟมากและยังโตต่อและเขตที่ใช้น้อยแต่โตเร็ว
- เขตที่ใช้ไฟมากและยังโตต่อ
  - ได้แก่ บางกะปิ, บางพลี, บางเขน, บางนา
- เขตที่ใช้ไฟน้อยแต่โตเร็ว
  - ได้แก่ มีนบุรี, บางใหญ่, ธนบุรี, บางบัวทอง, ลาดกระบัง

ซึ่งเขตเหล่านี้อาจจะเป็นเขตที่เป็น **พื้นที่เมืองที่กำลังขยายตัว**

<br>

**<u>Action</u>**:
- ภาครัฐควรมีการลงทุนโครงสร้างพื้นฐานเพิ่มเติมในบริเวณที่เป็นพื้นที่เมืองขยายตัว
- ภาคเอกชนสามารถลงทุนอสังหาริมทรัพย์ได้เพื่อรองรับการขยายตัวของประชากร

<br>

## <u>3. คำถามที่ 3</u> - รูปแบบการใช้ไฟฟ้าตามฤดูกาลต่างกันระหว่างเขตไหม Seasonal Pattern by District

- **โจทย์**: สร้างกราฟเปรียบเทียบรูปแบบฤดูกาลของการใช้ไฟฟ้ารายเดือนระหว่างเขตต่าง ๆ
- **สิ่งที่ต้องหาคำตอบ**: ระบุเขตที่มีความผันผวนตามฤดูกาลสูงผิดปกติและเขตที่การใช้ไฟฟ้าคงที่ พร้อมตั้งสมมติฐานลักษณะการใช้พลังงาน (เช่น เขตอุตสาหกรรม, พาณิชย์, ที่อยู่อาศัย) วิเคราะห์ความแตกต่างของลักษณะการใช้พลังงานในแต่ละพื้นที่
- **เครื่องมือวิเคราะห์ (KPI / Method)**: Seasonal Index, SD ของ Seasonal Index

<br>
<br>
<br>

| Visualization | ประเภทกราฟ | เป้าหมาย |
|--------------|-----------|----------|
| Visualization 3.1 | Heatmap (เดือน x เขต, Normalized) | เปรียบเทียบ seasonal shape ของทุกเขตพร้อมกัน |
| Visualization 3.2 | SD ของ Seasonal Index | จัดอันดับเขตตามความผันผวนของ Seasonal Index |

<br>
<br>
<br>


### Visualization 3.1 - Seasonal Heatmap: เดือน × เขต (Normalized)

**เหตุผลที่เลือก Heatmap ที่ Normalize แล้ว:**
แต่ละเขตมียอดการใช้ไฟต่างกันมาก การ Normalize เป็น Z-score ต่อเขต
ทำให้เปรียบเทียบ **รูปร่างของ Seasonal Pattern** โดยไม่ถูกรบกวนจากขนาดที่แตกต่างกัน
ช่องสีแดงเข้ม = เดือน Peak สูงกว่าค่าเฉลี่ยปกติของเขตนั้น, ช่องน้ำเงินเข้ม = ต่ำกว่า


**ดัชนีฤดูกาลรายเขต (Seasonal Index by District)**:
$$ \text{Seasonal Index}_{(\%)} = \left( \frac{\text{Average of District for the Month}}{\text{Grand Average of District}} \right) \times 100 $$


In [ ]:
# หาค่าเฉลี่ยรายเดือนของทุกเขต
seasonal_mean = df.groupby('เดือน_num')[DIST_COLS].mean()
# แทนที่ดัชนีที่เป็นตัวเลขด้วยชื่อเดือนภาษาไทยเพื่อให้แสดงผลบนกราฟ
seasonal_mean.index = [MONTH_TH[i] for i in seasonal_mean.index]

# คำนวณ Seasonal Index (SI) จาก Average for เดือน / Average of เขตตัวเองตลอดทั้งปี
grand_mean = df[DIST_COLS].mean()
seasonal_index = (seasonal_mean / grand_mean) * 100

print('คำนวณ Seasonal Index Heatmap เรียบร้อย')

คำนวณ Seasonal Index Heatmap เรียบร้อย


In [ ]:
# พล็อต Visualization 3.1
import plotly.express as px
fig = px.imshow(seasonal_index, color_continuous_scale='RdYlGn_r', color_continuous_midpoint=100,
                text_auto='.0f', aspect='auto')

fig.update_traces(textfont=dict(size=20))
fig.update_layout(title='<b>Visualization 3.1 - ดัชนีฤดูกาล (Seasonal Index) ของการใช้ไฟฟ้า 18 เขต<br>ตัวเลข > 100 หมายถึงใช้ไฟสูงกว่าค่าเฉลี่ยรวม (หน่วย: %)</b>',
                  margin=dict(t=120, b=80),
                  xaxis_title='<b>เขต</b>', yaxis_title='<b>เดือน</b>',
                  xaxis=dict(tickangle=-45, tickmode='linear'),
                  yaxis=dict(tickmode='linear', autorange='reversed'),
                  font=dict(family='TH Sarabun New', size=20), title_font_size=24)
fig.show()

### **<u>ตอบคำถาม</u>**
- อิทธิพลของฤดูกาลมีลักษณะคล้ายๆกัน ในทุกๆเขต

### Visualization 3.2 - S.D. ของ Seasonal Index: จัดอันดับความผันผวนของดัชนีฤดูกาล

**เหตุผลที่ใช้ S.D. ของ Seasonal Index**
- ค่า S.D. ใช้บอกการกระจายว่าโดยเฉลี่ยข้อมูลเบนออกจากค่าเฉลี่ยในระยะเท่าใด
- Seasonal Index บอกอิทธิพลของฤดูกาลในแต่ละเดือน ดังนั้นหาก Seasonal Index ผันผวนได้มากจะมี S.D. ที่สูง เนื่องจากขยับออกจากค่าเฉลี่ย Grand Mean ได้มาก
- ส่วนเขตที่ค่อนข้างมีความผันผวนต่ำหรือสามารถพยากรณ์หรือคาดเดาได้ง่ายจะมี S.D. ต่ำ เนื่องจากค่าขยับจาก Grand Mean ไม่มาก

In [ ]:
# ต้องการพล็อตจัดอันดับเขตตามค่าความผันผวนของดัชนีฤดูกาล (SD ของ Seasonal Index)

# คำนวณ Seasonal Index ของแต่ละเขตก่อน
seasonal_mean = df.groupby('เดือน_num')[DIST_COLS].mean()
grand_mean = df[DIST_COLS].mean()
seasonal_index = (seasonal_mean / grand_mean) * 100

#  หา Standard Deviation (SD) ของค่า SI 12 เดือน เพื่อวัดความผันผวนของอิทธิพลฤดูกาล
si_sd = seasonal_index.std().sort_values(ascending=False)

# สีตามระดับ SD: สูง = แดง (ผันผวนมาก), ต่ำ = เขียว (คงที่)
sd_colors = [
    '#E53935' if v >= si_sd.quantile(0.67) else
    '#FB8C00' if v >= si_sd.quantile(0.33) else '#43A047'
    for v in si_sd.values
]

tmp_df = pd.DataFrame({'เขต': si_sd.index[::-1], 'SD_SI': si_sd.values[::-1], 'color': sd_colors[::-1]})

print('คำนวณ SD ของ Seasonal Index เรียบร้อย')

คำนวณ SD ของ Seasonal Index เรียบร้อย


In [ ]:
# พล็อต Visualization 3.2 จัดอันดับเขตตามค่าความผันผวนของดัชนีฤดูกาล (SD ของ Seasonal Index)
fig = px.bar(tmp_df, x='SD_SI', y='เขต', color='color', color_discrete_map='identity', text_auto='.1f', orientation='h')
fig.update_traces(textfont_size=20, textposition='outside')
# ลากเส้นประโดยอิงจากค่ามัธยฐานของความผันผวนจากทุกเขตเพื่อใช้เปรียบเทียบระหว่างเขต
fig.add_vline(x=si_sd.median(), line_width=1.5, line_dash="dash", line_color="grey",
              annotation_text=f'Median SD = {si_sd.median():.1f}', annotation_font_size=18)

fig.update_layout(title='<b>Visualization 3.2 - ความผันผวนตามฤดูกาลรายเขต (วัดจากค่า SD ของ Seasonal Index)</b>',
                  margin=dict(t=120, r=60), height=800,
                  xaxis_title='<b>SD ของ Seasonal Index (ค่าสูง = ผันผวนฤดูกาลมาก)</b>', yaxis_title='<b>เขต</b>',
                  yaxis=dict(tickmode='linear', dtick=1),
                  font=dict(family='TH Sarabun New', size=20), title_font_size=24, showlegend=False)
fig.show()

### **<u>ตอบคำถาม</u>**

- **เขตที่มีค่า SD ของดัชนีฤดูกาลสูง (สะท้อนความผันผวนของการใช้ไฟฟ้ารายเดือนในรอบปีสูง)** ได้แก่  
  - เขตบางใหญ่  
  - เขตนวลจันทร์  
  - เขตบางบัวทอง  
  - เขตมีนบุรี  
  - เขตธนบุรี  
  - เขตบางเขน  

พื้นที่กลุ่มนี้มีรูปแบบการใช้ไฟฟ้าที่เปลี่ยนแปลงตามช่วงเวลาในรอบปี เมื่่อสังเกตุรายชื่อกลุ่มนี้จะเป็นย่านที่เป็นย่านที่อยู่อาศัยอย่างเห็นได้ชัด



---

- **เขตที่มีค่า SD ของดัชนีฤดูกาลในระดับปานกลาง (สะท้อนความผันผวนในระดับปกติ)** ได้แก่  
  - เขตนนทบุรี  
  - เขตบางขุนเทียน  
  - เขตลาดกระบัง  
  - เขตบางนา  
  - เขตบางรัก  
  - เขตสามเสน  
  - เขตยานนาวา  

พื้นที่กลุ่มนี้มีโครงสร้างการใช้ไฟฟ้าแบบผสม คือมีทั้งภาคที่อยู่อาศัย ธุรกิจพาริชย์ และภาคอุตสาหกรรมเล็กน้อย ทำให้ความผันผวนตามรอบฤดูกาลเกิดขึ้นไม่เต็มที่เมื่อเทียบกับกลุ่มแรก

---

- **เขตที่มีค่า SD ของดัชนีฤดูกาลต่ำ (สะท้อนความมีเสถียรภาพของการใช้ไฟฟ้าในรอบปี)** ได้แก่  
  - เขตราษฎร์บูรณะ  
  - เขตสมุทรปราการ  
  - เขตวัดเลียบ  
  - เขตบางพลี  
  - เขตคลองเตย  

พื้นที่กลุ่มนี้มีการใช้ไฟฟ้าค่อนข้างสม่ำเสมอตลอดทั้งปี จากรายชื่อจะเป็นบริเวณที่มีอุตสาหกรรมหรือภาคธุรกิจอยู่ มีสัดส่วนของที่อยู่อาศัยน้อยกว่า 2 กลุ่มก่อนหน้า

**กรณีพิเศษ**: ย่านราษฎร์บูรณะ ซึ่งเป็นย่านที่อยู่อาศัยแต่กลับมีความผันผวนด้านการใช้พลังงานต่ำ อย่างไรก็ตามย่านนี้อาจจะมีธุรกิจขนาดเล็ก หรือการใช้ไฟฟ้าเกิดขึ้นด้วยกิจกรรมพื้นฐานที่ฤดูกาลไม่ส่งผลกระทบนัก

---


<br>

**<u>Action</u>**:

- เขตที่การใช้ไฟฟ้าผันผวนสูงควรมีการสำรองพลังงานมากเป็นพิเศษสำหรับสถานการณ์ที่ไม่คาดคิด
- เขตที่เป็นย่านที่อยู่อาศัย ควรโฆษณาลดการใช้ไฟ เช่น รณรงค์ประหยัดไฟในฤดูร้อน หรือส่งเสริมเครื่องใช้ไฟฟ้าประหยัดพลังงาน
เขตที่มีโครงสร้างผสม (บ้าน + ธุรกิจ) ควรออกนโยบายแยกตามกลุ่มผู้ใช้ไฟ เช่น มาตรการสำหรับครัวเรือน และมาตรการสำหรับภาคธุรกิจ
- เขตที่มีพฤติกรรมต่างจากที่คาด เช่น ราษฎร์บูรณะ ควรศึกษาปัจจัยเพิ่มเติม เพราะอาจมี insight สำคัญที่นำไปใช้กับพื้นที่อื่นได้ (ขาดข้อมูล)
-

## <u>4. การวิเคราะห์ข้อมูลเพิ่มเติม</u> (นอกเหนือจากคำถาม Thackle)
จากการสำรวจลักษณะข้อมูล (EDA) และข้อคำถาม ต่อไปจะเป็นการวิเคราะห์เรื่องที่ต่อเนื่องกัน เพื่อตอบคำถามที่สนใจส่วนตัว

### คำถามส่วนตัว 4.1 - Stacked Area: สัดส่วนการใช้ไฟของแต่ละเขต

**เหตุผลที่เลือก Stacked Area:**
Stacked Area สามารถแสดง **ส่วนแบ่ง (Share)** ยอดใช้ของแต่ละเขตเทียบกับยอดรวม overtime

In [ ]:
# Visualization 4.1 - Stacked Area แสดงสัดส่วนการใช้ไฟแต่ละเขตด้วย Stacked Area plot

# คำนวณ Share การใช้ไฟ
df_pct = df[DIST_COLS].div(df['รวม'], axis=0) * 100
df_pct.index = df['date'].values
df_pct.sort_index(inplace=True)

# เลือก 8 อันดับแรก ที่เหลือมัดรวมเป็นอื่นๆ
N_TOP = 8
top_dist = avg_dist.nlargest(N_TOP).index.tolist()
others_pct = 100 - df_pct[top_dist].sum(axis=1)

tmp_df = df_pct[top_dist].copy()
tmp_df['เขตอื่น ๆ'] = others_pct
tmp_df.index.name = 'date'
tmp_df = tmp_df.reset_index().melt(id_vars='date', var_name='เขต', value_name='สัดส่วน (%)')

print('เตรียมข้อมูล Stacked Area เรียบร้อย')

เตรียมข้อมูล Stacked Area เรียบร้อย


In [ ]:
# Visualization 4.1 - Stacked Area: สัดส่วนการใช้ไฟของแต่ละเขต
fig = px.area(tmp_df, x='date', y='สัดส่วน (%)', color='เขต', color_discrete_sequence=px.colors.qualitative.Plotly + ['lightgrey'])
fig.update_layout(title=f'<b>Visualization 4.1 - สัดส่วนการใช้ไฟของแต่ละเขต (Top {N_TOP}) ตลอด {N_YEARS_TOTAL} ปี</b>',
                  margin=dict(t=120),
                  xaxis_title='<b>เดือน</b>', yaxis_title='<b>สัดส่วน (%)</b>',
                  font=dict(family='TH Sarabun New', size=20), title_font_size=24, yaxis_range=[0, 100])
fig.show()

### **<u>ผลลัพธ์การวิเคราะห์</u>**
- Share การใช้ไฟฟ้าของทุกเขตคงที่ทุกๆปีโดยจุดที่มีการลดลงพร้อมกันมาจากอิทธิพลของฤดูกาล โดยเขตที่ share มีการปรับตัวบ่อยจะได้รับอิทธิพลฤดูกาลเดือนต่างๆในรอบปีเยอะ ซึ่งจะเป็นเขตที่มีลักษณะเป็นย่านที่อยู่อาศัย

### คำถามส่วนตัว 4.2 - Correlation วิเคราะห์ความสัมพันธ์ระหว่างอุณภูมิเฉลี่ยต่อเดือนกับปริมาณการใช้ไฟฟ้า

การเก็บรวบรวมอุณภูมิเฉลี่ยต่อเดือนกรุงเทพ เก็บข้อมูลจากศูนย์ข้อมูลเปิดภาครัฐ
<br>
source: https://data.go.th/dataset/tmax-tmin

ข้อมูลมีลักษณะเก็บข้อมูลทุกจังหวัด เดือนละ 1 ไฟล์ CSV
ขั้นตอนการนำมาใช้
- preprocess data เอาข้อมูลเฉพาะคอลัมน์อุณหภูมิเฉลี่ยมาของแถวจังหวัดกรุงเทพมาจากหลายๆไฟล์มาเก็บไว้ใน 1 ไฟล์ CSV ที่มีข้อมูลอนุกรมเวลาอุณหภูมิเฉลี่ยรายเดือน
<br>

ข้อมูลอุณหภูมิเฉลี่ย หลัง preprocess เข้าถึงได้จาก

https://docs.google.com/spreadsheets/d/14vJc9aCzuzTzUr4E9AFINMeUXrymZSGuGYrtTZbRj98/edit?gid=458192458#gid=458192458

In [ ]:
# โหลดข้อมูลสภาพอากาศจากไฟล์ข้อมูลที่ได้เก็บรวบรวมมา (โหลดจาก Google Sheet)
weather_url = 'https://docs.google.com/spreadsheets/d/14vJc9aCzuzTzUr4E9AFINMeUXrymZSGuGYrtTZbRj98/export?format=csv&gid=458192458'
df_weather = pd.read_csv(weather_url, encoding='utf-8-sig')

# รวมตารางข้อมูลไฟฟ้าและข้อมูลอากาศเข้าด้วยกันโดยใช้ปีและเดือนเป็นตัวเชื่อม
df_analysis = df.merge(
    df_weather,
    left_on=['ปี_CE', 'เดือน_num'],
    right_on=['Year', 'Month'],
    how='inner'
)

In [ ]:
# กำหนดตัวแปรต้นคืออุณหภูมิสูงสุดและตัวแปรตามคือยอดรวมการใช้ไฟฟ้า
x_var = 'average_temp'
y_var = 'รวม'

# คำนวณค่าสหสัมพันธ์ของเพียร์สันเพื่อดูระดับความสัมพันธ์และค่าความเชื่อมั่นทางสถิติ
corr, p_value = stats.pearsonr(df_analysis[x_var], df_analysis[y_var])

# ตรวจสอบนัยสำคัญทางสถิติโดยใช้เกณฑ์มาตรฐานที่ระดับจุดศูนย์ห้า
if p_value < 0.05:
    result_text = "มีความสัมพันธ์กันอย่างมีนัยสำคัญทางสถิติ"
else:
    result_text = "ไม่มีความสัมพันธ์กันอย่างมีนัยสำคัญทางสถิติ"
print("p-value = ", p_value)
print(result_text)

p-value =  5.695116156186106e-20
มีความสัมพันธ์กันอย่างมีนัยสำคัญทางสถิติ


In [ ]:
# สร้างกราฟกระจายเพื่อแสดงการกระจายตัวของข้อมูลพร้อมเส้นแนวโน้มถดถอย
fig = px.scatter(
    df_analysis,
    x=x_var,
    y=y_var,
    trendline="ols",  # ใช้การคำนวณเส้นถดถอยแบบกำลังสองน้อยที่สุด
    hover_name='เดือน',
    title=f"<b>ความสัมพันธ์ระหว่าง {x_var} และยอดการใช้ไฟฟ้า</b><br>Correlation: {corr:.2f}",
    labels={x_var: 'อุณหภูมิเฉลี่ย', y_var: 'ยอดรวมการใช้ไฟฟ้า'},
    template='plotly_white'
)

# เพิ่ม Annotation เพื่อแสดงค่า p-value และผลลัพธ์บนกราฟ
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.95,
    text=f"p-value: {p_value:.4f}<br>({result_text})",
    showarrow=False,
    font=dict(family='TH Sarabun New', size=24, color="black"),
    align="center",
    bgcolor="rgba(255, 255, 255, 0.7)",
    bordercolor="red",
    borderwidth=1
)

# แสดงผลด้วย font ที่ลงไว้
fig.update_layout(
    font=dict(family='TH Sarabun New', size=20),
    title_font_size=30
)

fig.show()

### **<u>ผลลัพธ์การวิเคราะห์</u>**
- ความสัมพันธ์ระหว่างอุณหภูมิและปริมาณการใช้ฟ้าฟ้ามี Correlation = 0.8 มี p-value เข้าใกล้ 0 สรุปผลว่ามีความสัมพันธ์ในทางเดียวกันในระดับสูงมาก


<br>

**<u>Action</u>**:


- หากการพยกรณ์อากาศศทำได้อย่างแม่นยำ สามารถใช้ผลการพยากรณ์อากาศ มาพยากรณ์ปริมาณการใช้ไฟฟ้าขององค์กรในกรุงเทพได้เพื่อกำหนดนโยบายด้านพลังงาน
- องค์กรสามารถนำผลการพยากรณ์อากาศมาใช้ในการสร้างโมเดลในการพยากรณ์ปริมาณการใช้ไฟฟ้า เพื่อวางแผนสำรองเงินสำหรับค่าใช้จ่ายในส่วนนี้

# Actionable Insights

### **สรุปสิ่งที่พบและข้อเสนอแนะ (Findings & Actions)**

*   **รูปแบบฤดูกาลการใช้ไฟฟ้า:** ปริมาณการใช้ไฟฟ้ารวมมีจุดพีคในเดือนพฤษภาคม และต่ำสุดในเดือนธันวาคม
    *   **Action:**
        *   ควรมีการ stock ไฟฟ้าสำรองให้พอสำหรับช่วงเดือนพฤษภาคมเพื่อป้องกัน spike ผิดปกติ ส่วนในช่วงเดือนธ.ค. สามารถลดปริมาณการผลิตไฟฟ้าสำหรับเขตเหล่านี้ได้ ซึ่งอาจจะมีการนำปริมาณไฟฟ้าสำรองมาใช้เล็กน้อยเพื่อลดต้นทุนในการผลิตไฟฟ้า และช่วยลดค่า FT ของผู้บริโภค
        *   รัฐควรออกมาตรการค่าไฟเพื่อสำหรับประชาชนในช่วงเดือนมี.ค.-พ.ค. เนื่องจากอุณหภูมิประเทศไทยสูงขึ้นเรื่อยๆทุกปี ควรมีการลดความเหลื่อมล้ำในการเข้าถึงปริมาณพลังงาน เนื่องจากค่าไฟตอนนี้เก็บในอัตราก้าวหน้า
        *   สนับสนุนการใช้พลังงานหมุนเวียนในช่วงฤดูร้อนที่มีความต้องการพลังงานไฟฟ้าสูง

*   **เขตที่กำลังขยายตัวและเติบโต:** พบกลุ่มเขตที่ใช้ไฟมากและยังโตต่อ (บางกะปิ, บางพลี, บางเขน, บางนา) และเขตที่ใช้น้อยแต่โตเร็ว (มีนบุรี, บางใหญ่, ธนบุรี, บางบัวทอง, ลาดกระบัง)
    *   **Action:**
        *   ภาครัฐควรมีการลงทุนโครงสร้างพื้นฐานเพิ่มเติมในบริเวณที่เป็นพื้นที่เมืองขยายตัว
        *   ภาคเอกชนสามารถลงทุนอสังหาริมทรัพย์ได้เพื่อรองรับการขยายตัวของประชากร

*   **ความผันผวนตามฤดูกาลรายเขต:**
    *   **เขตที่มีค่า SD ของดัชนีฤดูกาลสูง (สะท้อนความผันผวนของการใช้ไฟฟ้ารายเดือนในรอบปีสูง)**: พื้นที่กลุ่มนี้มีรูปแบบการใช้ไฟฟ้าที่เปลี่ยนแปลงตามช่วงเวลาในรอบปี เมื่อสังเกตจะเป็นย่านที่เป็นย่านที่อยู่อาศัยอย่างเห็นได้ชัด
    *   **เขตที่มีค่า SD ของดัชนีฤดูกาลในระดับปานกลาง (สะท้อนความผันผวนในระดับปกติ)**: พื้นที่กลุ่มนี้มีโครงสร้างการใช้ไฟฟ้าแบบผสม คือมีทั้งภาคที่อยู่อาศัย ธุรกิจพาริชย์ และภาคอุตสาหกรรมเล็กน้อย ทำให้ความผันผวนตามรอบฤดูกาลเกิดขึ้นไม่เต็มที่เมื่อเทียบกับกลุ่มแรก
    *   **เขตที่มีค่า SD ของดัชนีฤดูกาลต่ำ (สะท้อนความมีเสถียรภาพของการใช้ไฟฟ้าในรอบปี)**: พื้นที่กลุ่มนี้มีการใช้ไฟฟ้าค่อนข้างสม่ำเสมอตลอดทั้งปี จากลักษณะจะเป็นบริเวณที่มีอุตสาหกรรมหรือภาคธุรกิจอยู่ มีสัดส่วนของที่อยู่อาศัยน้อยกว่า 2 กลุ่มก่อนหน้า
    *   **กรณีพิเศษ**: บางย่านที่เป็นที่อยู่อาศัยแต่กลับมีความผันผวนด้านการใช้พลังงานต่ำ อย่างไรก็ตามย่านนี้อาจจะมีธุรกิจขนาดเล็ก หรือการใช้ไฟฟ้าเกิดขึ้นด้วยกิจกรรมพื้นฐานที่ฤดูกาลไม่ส่งผลกระทบนัก
    *   **Action:**
        *   เขตที่การใช้ไฟฟ้าผันผวนสูงควรมีการสำรองพลังงานมากเป็นพิเศษสำหรับสถานการณ์ที่ไม่คาดคิด
        *   เขตที่เป็นย่านที่อยู่อาศัย ควรโฆษณาลดการใช้ไฟ เช่น รณรงค์ประหยัดไฟในฤดูร้อน หรือส่งเสริมเครื่องใช้ไฟฟ้าประหยัดพลังงาน
        *   เขตที่มีโครงสร้างผสม (บ้าน + ธุรกิจ) ควรออกนโยบายแยกตามกลุ่มผู้ใช้ไฟ เช่น มาตรการสำหรับครัวเรือน และมาตรการสำหรับภาคธุรกิจ
        *   เขตที่มีพฤติกรรมต่างจากที่คาด ควรศึกษาปัจจัยเพิ่มเติม เพราะอาจมี insight สำคัญที่นำไปใช้กับพื้นที่อื่นได้ (ขาดข้อมูล)

*   **ความสัมพันธ์กับอุณหภูมิ:** อุณหภูมิและปริมาณการใช้ฟ้าฟ้ามี Correlation = 0.8 มีความสัมพันธ์ในทางเดียวกันในระดับสูงมาก
    *   **Action:**
        * หากการพยากรณ์อากาศศทำได้อย่างแม่นยำ สามารถใช้ผลการพยากรณ์อากาศ มาพยากรณ์ปริมาณการใช้ไฟฟ้าขององค์กรในกรุงเทพได้เพื่อกำหนดนโยบายด้านพลังงาน
        *   องค์กรสามารถนำผลการพยากรณ์อากาศมาใช้ในการสร้างโมเดลในการพยากรณ์ปริมาณการใช้ไฟฟ้า เพื่อวางแผนสำรองเงินสำหรับค่าใช้จ่ายในส่วนนี้

# ขอบคุณครับ